In [1]:
import os
import sys
from pathlib import Path

if "__file__" in globals():
    project_root = Path(__file__).resolve().parent.parent
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if Path.cwd() != project_root:
    os.chdir(project_root)

In [2]:
import torch
from core.utils.config import load_config, print_config
from core.data.datamodule import BeatmapDataModule
from core.training import setup_device, create_kde_sampler
from core.training.pretrain import setup_pretraining, train
from core.model.bobert import BobertForPretraining

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {setup_device()}")
print(f"Working directory: {os.getcwd()}")

config = load_config("config", config_dir=".")
print_config(config, "Loaded BoBERT Configuration")

PyTorch version: 2.9.0+cu126
Using device: gpu
Working directory: /home/jessiez/projects/osu_corpora

--- Loaded BoBERT Configuration ---
data:
  max_seq_len: 2048
  val_split: 0.1
  num_workers: 4
  max_samples_per_class:
    aim: 2500
    tech: 2500
  min_stars: 4.0
  max_stars: 12.0
model:
  d_model: 512
  n_heads: 8
  n_layers: 6
  dim_feedforward_mult: 4
  dropout: 0.1
  local_attention_window: 256
components:
  use_flash_attention: true
  compile_model: true
  compile_mode: default
pretraining:
  db_path: ./data/beatmap_dataset_test/
  batch_size: 8
  num_epochs: 8
  learning_rate: 0.0002
  min_lr: 1.0e-06
  cooldown_type: cosine
  weight_decay: 0.05
  warmup_ratio: 0.1
  stable_ratio: 0.1
  use_amp: true
  checkpoint_dir: ./checkpoints
  grad_clip_norm: 1.0
  gradient_accumulation_steps: 8
  difficulty_loss_weight: 1.0
  mlm_loss_weight: 1.0
  masking_ratio: 0.25
  mean_span_length: 4
  sampling:
    method: kde
    kde_bandwidth: 0.5
    num_bins: 100
    strength: 0.2
finetuni

In [3]:
sampler_fn = lambda stars: create_kde_sampler(
    stars,
    bandwidth=config['pretraining']['sampling']['kde_bandwidth'],
    num_bins=config['pretraining']['sampling'].get('num_bins', 100),
    strength=config['pretraining']['sampling'].get('strength', 0.1),
)

datamodule = BeatmapDataModule(config, sampler_fn=sampler_fn)
datamodule.setup()

Loading raw data from Parquet dataset...
Loading beatmap metadata...
Found metadata for 19656 beatmaps. Processing in chunks of 5000...


Processing Chunks: 100%|██████████| 4/4 [00:23<00:00,  5.76s/it]


Loaded raw feature vectors for 19623 beatmaps.
Calculating missing difficulty attributes...


Calculating Attributes: 100%|██████████| 19/19 [00:00<00:00, 131721.94it/s]


Calculating normalization statistics...

                    NORMALIZATION STATISTICS

--- VECTOR STATISTICS:
------------------------------------------------------------
Field Name             Type         Param 1      Param 2     
------------------------------------------------------------
norm_x                 none         N/A          N/A         
norm_y                 none         N/A          N/A         
delta_x                mean/std     -0.0014      110.3529    
delta_y                mean/std     0.0029       98.8578     
log_time_diff_ms       mean/std     4.8668       0.4988      
bpm                    mean/std     184.8598     37.6173     
notes_per_second       mean/std     8.1638       2.7704      
velocity               mean/std     0.8968       0.8993      
relative_angle         none         N/A          N/A         
rhythm_change          mean/std     1.2052       4.1587      
log_slider_pixel_length mean/std     1.1705       2.0033      
slider_repeats         

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.fp32_precision = "tf32"
model = BobertForPretraining.from_config(config, device)

summary = model.bert.get_summary()
print(f"\n--- BERT Encoder Information ---")
print(f"Total Parameters: {summary['trainable_parameters'] / 1e6:.2f}M")
print(f"Model Dimension: {model.bert.d_model}")
print(f"Number of Heads: {model.bert.n_heads}")
print(f"Number of Layers: {model.bert.n_layers}")

/home/jessiez/projects/osu_corpora/.venv/lib/python3.12/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Compiling BERT pre-training model with torch.compile...

--- BERT Encoder Information ---
Total Parameters: 25.18M
Model Dimension: 512
Number of Heads: 8
Number of Layers: 6


In [5]:
module, trainer = setup_pretraining(config, datamodule.normalizer, model)

print(f"\nPretraining setup complete.")
print(f"Total epochs: {config['pretraining']['num_epochs']}")
print(f"Training samples: {len(datamodule.train_data)}")
print(f"Validation samples: {len(datamodule.val_data)}")

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores



Pretraining setup complete.
Total epochs: 8
Training samples: 17644
Validation samples: 1960


In [ ]:
train(module, trainer, datamodule)

print("\nBoBERT training completed!")

/home/jessiez/projects/osu_corpora/.venv/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/jessiez/projects/osu_corpora/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
/home/jessiez/projects/osu_corpora/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


Scheduler: WSD with 220 warmup, 220 stable, 1768 decay steps.
Cooldown type: cosine, Min LR Ratio: 0.0050


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ OptimizedModule │ 25.2 M │ train │     0 │
└───┴───────┴─────────────────┴────────┴───────┴───────┘

Trainable params: 25.2 M                                                                                           
Non-trainable params: 32                                                                                           
Total params: 25.2 M                                                                                               
Total estimated model params size (MB): 100                                                                        
Modules in train mode: 100                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

W1229 23:04:41.166000 119792 .venv/lib/python3.12/site-packages/torch/_inductor/utils.py:1558] [0/0_1] Not enough SMs to use max_autotune_gemm mode


Training: |          | 0/? [00:00<?, ?it/s]